In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp03/ex_8.py ──
"""
Shared infrastructure for MLFP03 Exercise 8 — Production ML + Drift +
Deployment.

Contains: data loading (Singapore credit scoring via MLFPDataLoader),
baseline LightGBM + isotonic calibration training, PSI/KS drift helpers,
and common output directory setup.

Technique-specific code (conformal quantile logic, dashboard rendering,
readiness checklist) stays in the per-technique files.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl  # noqa: F401  # re-exported for students
import lightgbm as lgb
from scipy import stats
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)

from kailash_ml import PreprocessingPipeline
from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "ex8_production"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore credit scoring (from MLFP02)
# ════════════════════════════════════════════════════════════════════════

RANDOM_SEED = 42


def load_credit_split() -> dict[str, Any]:
    """Load Singapore credit scoring data, preprocess, and return a split.

    Returns a dict with keys:
        X_train, y_train, X_test, y_test : numpy arrays
        feature_names                    : list[str]
        default_rate                     : float (train positive rate)
    """
    loader = MLFPDataLoader()
    credit = loader.load("mlfp02", "sg_credit_scoring.parquet")

    # Drop identifier columns before preprocessing — `customer_id` is a row
    # key, not a feature. Leaving it in the matrix causes drift noise to
    # dominate the top-variance feature list AND inflates AUC against the
    # model's pattern-recognition capability on IDs. Both are data-leakage
    # symptoms; the fix is to exclude the identifier at load time.
    id_columns = [c for c in ("customer_id", "application_id") if c in credit.columns]
    if id_columns:
        credit = credit.drop(id_columns)

    pipeline = PreprocessingPipeline()
    result = pipeline.setup(
        credit,
        target="default",
        seed=RANDOM_SEED,
        normalize=False,
        categorical_encoding="ordinal",
    )

    feature_cols = [c for c in result.train_data.columns if c != "default"]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_cols,
        target_column="default",
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_cols,
        target_column="default",
    )
    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "feature_names": col_info["feature_columns"],
        "default_rate": float(y_train.mean()),
    }


# ════════════════════════════════════════════════════════════════════════
# BASELINE MODEL — LightGBM + isotonic calibration
# ════════════════════════════════════════════════════════════════════════
#
# Hyperparameters come from Exercise 7's Bayesian optimisation run. These
# are frozen here so every technique file trains an identical baseline
# model and can be compared apples-to-apples.


def build_baseline_model(y_train: np.ndarray) -> lgb.LGBMClassifier:
    """Return an unfit LightGBM classifier configured for credit default."""
    base_rate = float(y_train.mean())
    scale_pos_weight = (1.0 - base_rate) / max(base_rate, 1e-6)
    return lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=7,
        num_leaves=63,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_SEED,
        verbose=-1,
    )


def train_calibrated_model(
    X_train: np.ndarray, y_train: np.ndarray
) -> CalibratedClassifierCV:
    """Train LightGBM and wrap with isotonic calibration (cv=5)."""
    base = build_baseline_model(y_train)
    base.fit(X_train, y_train)
    calibrated = CalibratedClassifierCV(base, method="isotonic", cv=5)
    calibrated.fit(X_train, y_train)
    return calibrated


def evaluate_classification(
    y_true: np.ndarray, y_proba: np.ndarray, threshold: float = 0.5
) -> dict[str, float]:
    """Return the full classification metric bundle used across ex_8."""
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred)),
        "auc_roc": float(roc_auc_score(y_true, y_proba)),
        "auc_pr": float(average_precision_score(y_true, y_proba)),
        "log_loss": float(log_loss(y_true, y_proba)),
        "brier": float(brier_score_loss(y_true, y_proba)),
    }


# ════════════════════════════════════════════════════════════════════════
# DRIFT STATISTICS — PSI + KS
# ════════════════════════════════════════════════════════════════════════
#
# PSI (Population Stability Index):
#     PSI = Σ (p_new - p_ref) * ln(p_new / p_ref)
#     < 0.1 no shift, 0.1-0.2 moderate, > 0.2 significant
# KS (Kolmogorov-Smirnov): two-sample test on the empirical CDFs.


def compute_psi(reference: np.ndarray, current: np.ndarray, bins: int = 10) -> float:
    """Compute Population Stability Index on 1-D arrays."""
    _, bin_edges = np.histogram(reference, bins=bins)
    ref_counts, _ = np.histogram(reference, bins=bin_edges)
    cur_counts, _ = np.histogram(current, bins=bin_edges)
    # Laplace-smooth to avoid log(0)
    ref_props = (ref_counts + 1) / (len(reference) + bins)
    cur_props = (cur_counts + 1) / (len(current) + bins)
    return float(np.sum((cur_props - ref_props) * np.log(cur_props / ref_props)))


def compute_ks(reference: np.ndarray, current: np.ndarray) -> tuple[float, float]:
    """Return (KS statistic, p-value) on 1-D arrays."""
    ks_stat, p_value = stats.ks_2samp(reference, current)
    return float(ks_stat), float(p_value)


def drift_row(
    reference: np.ndarray, current: np.ndarray, psi_threshold: float = 0.1
) -> dict[str, float | str]:
    """Return {psi, ks_stat, ks_pval, drift} for a single feature."""
    psi = compute_psi(reference, current)
    ks_stat, ks_pval = compute_ks(reference, current)
    drift = "YES" if (psi > psi_threshold or ks_pval < 0.05) else "No"
    return {"psi": psi, "ks_stat": ks_stat, "ks_pval": ks_pval, "drift": drift}


def simulate_gradual_drift(
    X_ref: np.ndarray, X_new: np.ndarray, n_features: int = 3, shift: float = 0.5
) -> np.ndarray:
    """Return a copy of X_new with mean-shifted top features (drift sim)."""
    drifted = X_new.copy()
    for i in range(n_features):
        drifted[:, i] += shift * X_ref[:, i].std()
    return drifted


def simulate_sudden_drift(
    X_ref: np.ndarray,
    X_new: np.ndarray,
    feature_idx: int = 0,
    sigma_shift: float = 3.0,
    seed: int = RANDOM_SEED,
) -> np.ndarray:
    """Replace one column in X_new with a heavily shifted Gaussian."""
    rng = np.random.default_rng(seed)
    drifted = X_new.copy()
    drifted[:, feature_idx] = rng.normal(
        loc=X_ref[:, feature_idx].mean() + sigma_shift * X_ref[:, feature_idx].std(),
        scale=X_ref[:, feature_idx].std(),
        size=drifted.shape[0],
    )
    return drifted




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 8.2: Drift Monitoring with PSI and KS
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Configure a drift spec with PSI and KS thresholds
#   - Compute PSI/KS against a reference distribution
#   - Simulate gradual and sudden drift and verify detection
#   - Measure the AUC degradation that drift causes on a live model
#   - Translate drift alerts into a retraining decision with $ impact
#
# PREREQUISITES: Exercise 8.1 (you need the calibrated model).
#
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Theory     — why drift is the #1 cause of production ML failure
#   2. Build      — PSI + KS helpers and the DriftSpec configuration
#   3. Train      — establish a drift baseline from train vs test
#   4. Visualise  — feature-level drift heatmap + performance degradation
#   5. Apply      — MAS retraining rule for an SG consumer-credit model
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
from sklearn.metrics import roc_auc_score


from kailash_ml.engines.drift_monitor import DriftSpec



## THEORY — Why drift is the silent killer of production ML


In [ ]:
# A model is trained on a photograph of the world. The world keeps moving.
#
# COVARIATE DRIFT:  p(x) changes. Interest rates rose 200 bps, customers'
#                   debt-service ratios shifted. Same labels, new inputs.
# LABEL DRIFT:      p(y) changes. Unemployment spiked, default rate jumped
#                   from 12% to 17%. Same inputs, new base rate.
# CONCEPT DRIFT:    p(y | x) changes. The RELATIONSHIP shifts. The same
#                   debt ratio that was safe last year is risky this year.
#
# Every drift flavour destroys model performance silently. You'll only
# notice if (a) you have drift monitoring, or (b) three months later the
# quarterly review surfaces a 200 bps AUC collapse.
#
# TWO COMPLEMENTARY DRIFT TESTS:
#
# PSI (Population Stability Index)
#   PSI = Σ (p_new - p_ref) * ln(p_new / p_ref)
#   - < 0.1  no meaningful shift
#   - 0.1-0.2 moderate shift, investigate
#   - > 0.2  significant shift, retrain
#   Strength: interpretable, industry standard in credit scoring.
#
# KS (Kolmogorov-Smirnov two-sample test)
#   Max distance between empirical CDFs, with a p-value.
#   Strength: catches localised CDF changes PSI's bins miss.
#
# Use BOTH. PSI catches broad distribution shifts, KS catches sharp local
# ones. Alerting when EITHER trips gives the tightest safety net.



## TASK 2 — BUILD: drift helpers + DriftSpec


In [ ]:
print("\n" + "=" * 70)
print("  MLFP03 Exercise 8.2 — Drift Monitoring")
print("=" * 70)

split = load_credit_split()
X_train, y_train = split["X_train"], split["y_train"]
X_test, y_test = split["X_test"], split["y_test"]
feature_names = split["feature_names"]

# Train the same calibrated model as 8.1 so we can measure how drift
# damages ITS performance specifically.
calibrated_model = train_calibrated_model(X_train, y_train)
y_proba_ref = calibrated_model.predict_proba(X_test)[:, 1]
auc_ref = float(roc_auc_score(y_test, y_proba_ref))
print(f"\nReference model AUC-ROC: {auc_ref:.4f}")

# DriftSpec is the scheduled-monitoring contract for kailash-ml's
# DriftMonitor. The thresholds override the monitor-wide defaults for
# this specific schedule; `feature_columns=None` means "all features
# from the stored reference". An `on_drift_detected` async callback can
# be wired in when you want to page on-call instead of just logging.
drift_spec = DriftSpec(
    psi_threshold=0.1,
    ks_threshold=0.05,
)
print(
    f"\nDriftSpec configured: PSI>{drift_spec.psi_threshold}, "
    f"KS p-value<{drift_spec.ks_threshold} (daily cadence set at the "
    f"DriftMonitor.schedule call-site, not on the spec itself)"
)



### Checkpoint 1


In [ ]:
assert auc_ref > 0.5, "Task 2: Reference AUC should beat random"
print("\n[ok] Checkpoint 1 — drift spec + reference model ready\n")



## TASK 3 — TRAIN: establish the drift baseline (train vs test)


In [ ]:
# If train vs test shows high PSI, the split itself is broken — you
# can't monitor drift against a poisoned baseline.

print("=== Baseline drift (train vs test — same distribution) ===")
print(f"{'Feature':<30} {'PSI':>8} {'KS stat':>10} {'KS p-val':>10} {'Drift?':>8}")
print("─" * 70)

baseline_drift: dict[str, dict[str, float | str]] = {}
for i, feat in enumerate(feature_names[:10]):
    row = drift_row(X_train[:, i], X_test[:, i])
    baseline_drift[feat] = row
    print(
        f"  {feat:<28} {row['psi']:>8.4f} {row['ks_stat']:>10.4f} "
        f"{row['ks_pval']:>10.4f} {row['drift']:>8}"
    )



### Checkpoint 2


In [ ]:
assert len(baseline_drift) > 0, "Task 3: Baseline drift should be populated"
baseline_alerts = sum(1 for r in baseline_drift.values() if r["drift"] == "YES")
assert (
    baseline_alerts <= 3
), f"Task 3: Train vs test should not drift on most features (got {baseline_alerts})"
print(f"\n[ok] Checkpoint 2 — baseline has {baseline_alerts} false-alert features\n")



## TASK 4 — VISUALISE drift + performance degradation


In [ ]:
# Simulate two drift scenarios and watch the reference model's AUC drop.

# Scenario A: GRADUAL drift — 0.5σ shift on top 3 features.
X_gradual = simulate_gradual_drift(X_train, X_test, n_features=3, shift=0.5)

print("=== Scenario A: Gradual drift (0.5σ mean shift on top 3 features) ===")
print(f"{'Feature':<30} {'PSI':>8} {'KS stat':>10} {'KS p-val':>10} {'Alert?':>10}")
print("─" * 72)

gradual_psis = []
for i in range(5):
    feat = feature_names[i]
    psi = compute_psi(X_train[:, i], X_gradual[:, i])
    ks_stat, ks_pval = compute_ks(X_train[:, i], X_gradual[:, i])
    alert = "YES (!)" if (psi > 0.1 or ks_pval < 0.05) else "No"
    gradual_psis.append(psi)
    print(f"  {feat:<28} {psi:>8.4f} {ks_stat:>10.4f} {ks_pval:>10.4f} {alert:>10}")

y_proba_gradual = calibrated_model.predict_proba(X_gradual)[:, 1]
auc_gradual = float(roc_auc_score(y_test, y_proba_gradual))
print(f"\nAUC under gradual drift: {auc_gradual:.4f} (baseline {auc_ref:.4f})")
print(f"Degradation: {auc_ref - auc_gradual:.4f}")

# Scenario B: SUDDEN drift — swap feature 0 for a 3σ-shifted Gaussian.
X_sudden = simulate_sudden_drift(X_train, X_test, feature_idx=0, sigma_shift=3.0)
psi_sudden = compute_psi(X_train[:, 0], X_sudden[:, 0])
ks_stat_sudden, ks_pval_sudden = compute_ks(X_train[:, 0], X_sudden[:, 0])

print(f"\n=== Scenario B: Sudden drift (3σ mean shift on feature 0) ===")
print(f"  {feature_names[0]}: PSI={psi_sudden:.4f}, KS p-val={ks_pval_sudden:.2e}")
y_proba_sudden = calibrated_model.predict_proba(X_sudden)[:, 1]
auc_sudden = float(roc_auc_score(y_test, y_proba_sudden))
print(f"  AUC under sudden drift: {auc_sudden:.4f} (Δ={auc_ref - auc_sudden:.4f})")

# Visualise: three-panel stacked bar + AUC degradation line.
fig = go.Figure()
x_labels = [feature_names[i] for i in range(5)]
fig.add_trace(
    go.Bar(
        x=x_labels,
        y=[float(compute_psi(X_train[:, i], X_test[:, i])) for i in range(5)],
        name="PSI (baseline)",
        marker_color="#10b981",
    )
)
fig.add_trace(
    go.Bar(
        x=x_labels,
        y=gradual_psis,
        name="PSI (gradual drift)",
        marker_color="#f59e0b",
    )
)
fig.add_trace(
    go.Bar(
        x=x_labels,
        y=[psi_sudden]
        + [float(compute_psi(X_train[:, i], X_sudden[:, i])) for i in range(1, 5)],
        name="PSI (sudden drift)",
        marker_color="#ef4444",
    )
)
fig.add_hline(
    y=0.1, line_dash="dash", line_color="#64748b", annotation_text="PSI=0.1 (moderate)"
)
fig.add_hline(
    y=0.2, line_dash="dot", line_color="#b91c1c", annotation_text="PSI=0.2 (retrain)"
)
fig.update_layout(
    title="Feature-level PSI across drift scenarios (Singapore credit)",
    xaxis_title="Feature",
    yaxis_title="PSI",
    barmode="group",
    height=520,
)
viz_path = OUTPUT_DIR / "ex8_02_drift_psi.html"
fig.write_html(str(viz_path))
print(f"\nSaved: {viz_path}")



### Checkpoint 3


In [ ]:
# The contract we verify is "drift is DETECTED" (PSI breach + KS p-value),
# not "AUC always collapses". On a well-separable dataset the model may
# still ride other features through the drift — which is exactly why the
# drift-detection layer is INDEPENDENT from the performance layer. Both
# signals must be watched in production. A tiny tolerance absorbs
# Monte-Carlo noise on the sudden-drift Gaussian simulator.
assert psi_sudden > 0.1, "Task 4: Sudden drift should produce PSI > 0.1"
assert ks_pval_sudden < 0.05, "Task 4: Sudden drift should be detected by KS"
assert (
    auc_sudden <= auc_ref + 0.005
), "Task 4: Drift should not materially improve AUC (beyond MC noise)"
print("\n[ok] Checkpoint 3 — drift detected (perf layer watches it separately)\n")



## TASK 5 — APPLY: MAS retraining rule for SG consumer credit


In [ ]:
# SCENARIO: OCBC Bank Singapore runs an unsecured credit-card application
# model scoring ~45,000 applications/month. MAS Notice 635 (§24) requires
# "ongoing validation of credit decision models" — but doesn't say HOW.
#
# The informal quarterly review catches major drift ~60 days after it
# starts. During those 60 days the model is making bad decisions:
#
#   - False approvals cost S$4,200 each (median unsecured LGD × loan size)
#   - False rejections cost S$180 each (opportunity cost on declined NPV)
#   - Without drift monitoring: ~3% AUC drop → ~4,500 bad decisions/quarter
#     → ~S$5.2M/quarter in avoidable losses.
#
# With daily PSI + KS monitoring:
#   - Alert triggers on day 1-3 of meaningful drift
#   - Retrain pipeline runs within the week
#   - 90% of the loss window is eliminated
#   - Estimated saving: S$4.7M/quarter = S$19M/year.

monthly_apps = 45_000
bad_decision_cost_sgd = 4_200.0  # false-approval LGD cost
quarterly_bad_without_monitoring = monthly_apps * 3 * 0.033  # ~3.3% error rate jump
loss_window_days_informal = 60
loss_window_days_monitored = 7
savings_ratio = 1 - (loss_window_days_monitored / loss_window_days_informal)

loss_without = quarterly_bad_without_monitoring * bad_decision_cost_sgd
loss_with = loss_without * (1 - savings_ratio)
savings_q = loss_without - loss_with

print(f"\n=== OCBC retraining economics (45K apps/mo) ===")
print(f"  Quarterly loss without monitoring:  S${loss_without:>14,.0f}")
print(f"  Quarterly loss with daily monitor:  S${loss_with:>14,.0f}")
print(f"  Savings:                             S${savings_q:>14,.0f}/quarter")
print(f"  Annualised:                          S${savings_q * 4:>14,.0f}/year")
print("\n  RETRAINING RULE (production):")
print("    IF  any_feature_PSI > 0.2  OR  live_AUC_PR < 0.9 * baseline_AUC_PR")
print("    THEN queue retraining, freeze auto-decisions, notify risk lead")



## REFLECTION


[x] Configured DriftSpec with PSI + KS thresholds
  [x] Established a baseline from train vs test ({baseline_alerts} false alerts)
  [x] Simulated gradual and sudden drift, caught both
  [x] Measured AUC degradation: {auc_ref:.3f} → {auc_sudden:.3f} under sudden drift
  [x] Translated drift alerts into a quantified retraining rule
      (~S${savings_q * 4:,.0f}/yr at OCBC scale)

  KEY INSIGHT: The pipeline that catches drift earns its keep the week
  the world changes. Quarterly reviews are insurance for yesterday.

  Next: 03_model_card.py — document every decision this model makes
  so the regulator can reconstruct WHY when the drift alert fires.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

